In [ ]:
import pandas as pd
import pyarrow.parquet as pq
import sys
import os
import numpy as np
import random as r


sys.path.insert(0, os.path.join(os.getcwd(), 'chronoepilogi_implementation'))

# Now import the class
from ce_extensions2 import ChronoEpilogi

In [ ]:
###ONE HOT ENCODING OF THE COMPONENT

db = pd.read_parquet("RUL_FRMTM.parquet", engine="fastparquet")
##Cleaning of all the row having a nan for a columns 
##One-Hot encoding 
column_to_OHE="Component"
db_encoded = pd.get_dummies(db, prefix="Component_", columns=[column_to_OHE], dtype=int)

#########################################################
# THIS CAN BE USED TO REDUCE THE DATASET SIZE IF NEEDED
non_na_rows = db[~db["System"].isna()]
na_rows = db[db["System"].isna()].sample(n=3000, random_state=42)  # Adjust 'n' as needed
db_sampled = pd.concat([non_na_rows, na_rows])
db_sampled.sort_values(by=["Date_Time", "EPISODE_ID"], inplace=True)
db=db_sampled.copy()
##########################################################

db_encoded = pd.get_dummies(db_sampled, prefix="Class_", columns=[column_to_OHE])
print(f"Size of the sampled database: {db_encoded.shape}")

target = 'RUL'  #target column
target_type =  "continuous"
column_names = db_encoded.columns.tolist()


## we take all the columns wanted for the model
numerical_columns = [col for col in column_names if db_encoded[col].dtype == np.float64 and col != target]
categorical_columns = [col for col in column_names if col.startswith("Class_")]

## and add them to our features database
features_db = numerical_columns + categorical_columns + [target]


#X = pd.concat([features_db, db_encoded[target]], axis=1)
X = db_encoded[features_db].copy()

## To prevent dtype issue, we set the type to a suitable one

X[numerical_columns] = X[numerical_columns].astype(np.float64)
X[categorical_columns] = X[categorical_columns].astype(np.int64)

## Cleaning the NaN and the +- inf
for col in numerical_columns:
    X[col] = X[col].fillna(X[col].mean())

## Data summary
print(f"Keeping {len(numerical_columns)+len(categorical_columns)+len(target)} columns:")
print(f"Categorical columns: {len(categorical_columns)}")
print(f"Numerical columns: {len(numerical_columns)}")
print(f"Target:{target}")


variable_types = {
    **{col: "numerical" for col in numerical_columns},
    **{col: "categorical" for col in categorical_columns}
}
